In [1]:
import pandas as pd
import numpy as np
import sklearn
import plotly

In [2]:
df = pd.read_csv("steam_reviews 1.csv")
df

,date_posted,funny,helpful,hour_played,is_early_access_review,recommendation,review,title
0,2018-11-24,0,0,40,False,Recommended,10/10 would murder kids again,Rust
1,2016-06-28,0,0,20,False,Recommended,OH YEAH,Euro Truck Simulator 2
2,2016-01-09,0,0,292,True,Recommended,is good gam,Rust
3,2018-07-08,0,0,401,False,Recommended,Spooky.,Dead by Daylight
4,2018-12-29,1,1,125,False,Not Recommended,Tries to play for an hour with my bros.Had to ...,Grand Theft Auto V
...,...,...,...,...,...,...,...,...
14612,2018-01-26,0,0,369,False,Recommended,FUNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN,PLAYERUNKNOWN'S BATTLEGROUNDS
14613,2017-06-19,0,0,1442,False,Recommended,just play,Rocket League®
14614,2018-06-13,0,0,64,False,Recommended,Trust Me,Terraria
14615,2018-11-21,0,0,408,False,Recommended,Pretty good job so far. Pretty fun game 11/10 ...,Dead by Daylight


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14617 entries, 0 to 14616
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   date_posted             14617 non-null  str  
 1   funny                   14617 non-null  int64
 2   helpful                 14617 non-null  int64
 3   hour_played             14617 non-null  int64
 4   is_early_access_review  14617 non-null  bool 
 5   recommendation          14617 non-null  str  
 6   review                  14610 non-null  str  
 7   title                   14617 non-null  str  
dtypes: bool(1), int64(3), str(4)
memory usage: 4.7 MB


In [4]:
df.isna().sum()

date_posted               0
funny                     0
helpful                   0
hour_played               0
is_early_access_review    0
recommendation            0
review                    7
title                     0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df["title"].nunique()

23

23 Unique games to score

Number of reviews per game/credibility

Should impact confidence of scoring

In [7]:
df["title"].value_counts(normalize=True)

title
PLAYERUNKNOWN'S BATTLEGROUNDS                  0.173702
Rust                                           0.156735
Grand Theft Auto V                             0.148731
Rocket League®                                 0.121297
Dead by Daylight                               0.070740
The Elder Scrolls V: Skyrim Special Edition    0.066840
MONSTER HUNTER: WORLD                          0.052952
ASTRONEER                                      0.039475
RESIDENT EVIL 2 / BIOHAZARD RE:2               0.033933
Euro Truck Simulator 2                         0.022645
Terraria                                       0.016351
Slay the Spire                                 0.015804
Insurgency: Sandstorm                          0.014230
Subnautica                                     0.013956
Left 4 Dead 2                                  0.013820
RimWorld                                       0.012588
Stardew Valley                                 0.012520
Factorio                                  

POsitive vs negative review balance by game based on recommended

In [8]:
recommendation_by_game = (
    df.assign(is_recommended=df["recommendation"].eq("Recommended"))
      .groupby("title")["is_recommended"]
      .mean()
      .sort_values(ascending=False)
)
recommendation_by_game

title
ACE COMBAT™ 7: SKIES UNKNOWN                   1.000000
Beat Saber                                     1.000000
Euro Truck Simulator 2                         1.000000
Factorio                                       1.000000
Warhammer 40,000: Mechanicus                   1.000000
Terraria                                       1.000000
RimWorld                                       1.000000
Subnautica                                     1.000000
Stardew Valley                                 1.000000
Left 4 Dead 2                                  0.995050
Slay the Spire                                 0.991342
RESIDENT EVIL 2 / BIOHAZARD RE:2               0.985887
Insurgency: Sandstorm                          0.942308
Overcooked! 2                                  0.900000
ASTRONEER                                      0.861352
PLAYERUNKNOWN'S BATTLEGROUNDS                  0.683340
Rust                                           0.664339
The Elder Scrolls V: Skyrim Special Editio

Playtime distribution by game

In [9]:
df["hour_played"].describe()
#np.log1p(df["hour_played"]).describe()

count    14617.000000
mean       296.143395
std        510.080218
min          0.000000
25%         33.000000
50%        122.000000
75%        340.000000
max       9567.000000
Name: hour_played, dtype: float64

## Helpful and funny review engagement

In [10]:
print((df["helpful"] == 0).mean())
print((df["funny"] == 0).mean())

0.9022371211602928
0.9022371211602928


In [11]:
df.groupby("recommendation")[["helpful", "funny", "hour_played"]].median()

,helpful,funny,hour_played
recommendation,,,
Not Recommended,0.0,0.0,95.0
Recommended,0.0,0.0,133.0


### Helpful/funny overlap

Before using `helpful` and `funny` as separate modelling features, we need to check whether they provide independent information. The counts themselves are not identical, but the binary flags `helpful > 0` and `funny > 0` are perfectly aligned in this dataset.

In [12]:
df["has_helpful_vote"] = df["helpful"] > 0
df["has_funny_vote"] = df["funny"] > 0

helpful_funny_overlap = pd.crosstab(
    df["has_helpful_vote"],
    df["has_funny_vote"],
    rownames=["helpful > 0"],
    colnames=["funny > 0"],
)

helpful_funny_flag_correlation = df[
    ["has_helpful_vote", "has_funny_vote"]
].corr().iloc[0, 1]

print(f"Correlation between helpful/funny vote flags: {helpful_funny_flag_correlation:.3f}")
helpful_funny_overlap

Correlation between helpful/funny vote flags: 1.000


funny > 0,False,True
helpful > 0,,
False,13188,0
True,0,1429


Because the positive-vote flags are perfectly correlated, using both would double-count the same review engagement signal. For the game-level modelling table, we keep `helpful` because it is more directly tied to review usefulness for acquisition decisions, and drop the funny-derived aggregate features.

Review activity over time

In [13]:
df["date_posted"] = pd.to_datetime(df["date_posted"])

game_time = df.groupby("title").agg(
    first_review=("date_posted", "min"),
    last_review=("date_posted", "max"),
    review_count=("review", "count")
)

game_time["review_span_days"] = (
    game_time["last_review"] - game_time["first_review"]
).dt.days
game_time

,first_review,last_review,review_count,review_span_days
title,,,,
ACE COMBAT™ 7: SKIES UNKNOWN,2019-01-31,2019-02-05,8,5
ASTRONEER,2016-12-15,2019-02-13,576,790
Beat Saber,2018-05-04,2018-12-24,10,234
Dead by Daylight,2016-06-14,2019-02-09,1034,970
Euro Truck Simulator 2,2013-02-15,2017-07-25,330,1621
Factorio,2016-02-25,2018-10-01,158,949
Farming Simulator 19,2018-11-19,2019-01-20,7,62
Grand Theft Auto V,2015-04-13,2019-02-16,2173,1405
Insurgency: Sandstorm,2018-12-12,2019-01-27,208,46


In [14]:
df["date_posted"] = pd.to_datetime(df["date_posted"])

time_features = df.groupby("title").agg(
    first_review=("date_posted", "min"),
    last_review=("date_posted", "max"),
    review_span_days=("date_posted", lambda x: (x.max() - x.min()).days),
)
time_features

,first_review,last_review,review_span_days
title,,,
ACE COMBAT™ 7: SKIES UNKNOWN,2019-01-31,2019-02-05,5
ASTRONEER,2016-12-15,2019-02-13,790
Beat Saber,2018-05-04,2018-12-24,234
Dead by Daylight,2016-06-14,2019-02-09,970
Euro Truck Simulator 2,2013-02-15,2017-07-25,1621
Factorio,2016-02-25,2018-10-01,949
Farming Simulator 19,2018-11-19,2019-01-20,62
Grand Theft Auto V,2015-04-13,2019-02-16,1405
Insurgency: Sandstorm,2018-12-12,2019-01-27,46


In [15]:
df["is_early_access_review"].value_counts(normalize=True)
df.groupby("is_early_access_review")["recommendation"].value_counts(normalize=True)

is_early_access_review  recommendation 
False                   Recommended        0.680855
                        Not Recommended    0.319145
True                    Recommended        0.785259
                        Not Recommended    0.214741
Name: proportion, dtype: float64

## Create a game-level modelling table

The raw dataset is review-level, but GameVault makes decisions at game level. The next step aggregates all reviews for each `title` into one row per game. This gives us the table we can use for EDA, unsupervised scoring, clustering, and dashboard ranking.


7 missing reviews, fill "" neutral

In [16]:
# Basic review-level preparation
df["date_posted"] = pd.to_datetime(df["date_posted"], errors="coerce")
df["review"] = df["review"].fillna("")
df["is_recommended"] = df["recommendation"].eq("Recommended")
df["review_length"] = df["review"].str.len()

total_reviews = len(df)
total_helpful_votes = df["helpful"].sum()
total_reviews, total_helpful_votes

(14617, np.int64(54310))

## Review length in lexical word units

Before choosing a sentiment method, check review length using word-like tokens rather than characters. VADER is most appropriate for short, informal review text, so this helps justify the sentiment approach.

In [17]:
review_token_pattern = r"[A-Za-z']+"

df["review_word_count"] = (
    df["review"]
    .fillna("")
    .str.findall(review_token_pattern)
    .str.len()
)

review_word_count_summary = df["review_word_count"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

print(f"Maximum review length: {df['review_word_count'].max()} word tokens")
review_word_count_summary

Maximum review length: 1481 word tokens


count    14617.000000
mean        43.934049
std         85.868855
min          0.000000
25%          6.000000
50%         16.000000
75%         46.000000
90%        107.000000
95%        172.000000
99%        409.520000
max       1481.000000
Name: review_word_count, dtype: float64

## VADER sentiment scoring

The token-length check above shows that many reviews are short, informal player comments rather than long documents. For this reason, use VADER (`vaderSentiment`), a lexicon and rule-based sentiment analyser designed for short, informal text such as social media posts and user reviews. This is stronger than simply counting positive and negative words because VADER adjusts for negation, intensifiers, punctuation, capitalisation, slang, emoticons, and contrastive phrasing. The main score is `compound`, which ranges from -1 to 1.

In [18]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()


def vader_sentiment(text):
    if not isinstance(text, str) or not text.strip(): # for my NaNs encoded as "" taken as neutral
        return {
            "sentiment_neg": 0.0,
            "sentiment_neu": 1.0,
            "sentiment_pos": 0.0,
            "sentiment_score": 0.0,
            "sentiment_label": "neutral",
        }

    scores = vader.polarity_scores(text)
    compound = scores["compound"]

    if compound >= 0.05:
        label = "positive"
    elif compound <= -0.05:
        label = "negative"
    else:
        label = "neutral"

    return {
        "sentiment_neg": scores["neg"],
        "sentiment_neu": scores["neu"],
        "sentiment_pos": scores["pos"],
        "sentiment_score": compound,
        "sentiment_label": label,
    }


sentiment_columns = pd.DataFrame(
    df["review"].map(vader_sentiment).tolist(),
    index=df.index,
)

df = pd.concat([df, sentiment_columns], axis=1)

df[
    [
        "title",
        "recommendation",
        "review",
        "sentiment_neg",
        "sentiment_neu",
        "sentiment_pos",
        "sentiment_score",
        "sentiment_label",
    ]
].head()

,title,recommendation,review,sentiment_neg,sentiment_neu,sentiment_pos,sentiment_score,sentiment_label
0,Rust,Recommended,10/10 would murder kids again,0.540,0.460,0.000,-0.6908,negative
1,Euro Truck Simulator 2,Recommended,OH YEAH,0.000,0.312,0.688,0.2960,positive
2,Rust,Recommended,is good gam,0.000,0.408,0.592,0.4404,positive
3,Dead by Daylight,Recommended,Spooky.,0.000,1.000,0.000,0.0000,neutral
4,Grand Theft Auto V,Not Recommended,Tries to play for an hour with my bros.Had to ...,0.177,0.734,0.089,-0.7922,negative


## Validate sentiment against recommendations

The Steam recommendation label is not the same as text sentiment, but they should be directionally related. This is a validation check, not a requirement that VADER perfectly predicts recommendations. If VADER struggles with `Not Recommended` reviews, that tells us to use text sentiment as a supporting signal rather than as the main approval metric.

From the below we see that majority of recommended=1 reviews are marked as positive, supporting the analysis

In [19]:
sentiment_recommendation_table = pd.crosstab(
    df["recommendation"],
    df["sentiment_label"],
    normalize="index",
).round(3)

sentiment_recommendation_table

sentiment_label,negative,neutral,positive
recommendation,,,
Not Recommended,0.440,0.166,0.394
Recommended,0.111,0.131,0.758


In [20]:
df["recommendation_sentiment_agrees"] = np.select(
    [
        (df["recommendation"].eq("Recommended") & df["sentiment_label"].eq("positive")),
        (df["recommendation"].eq("Not Recommended") & df["sentiment_label"].eq("negative")),
    ],
    [True, True],
    default=False,
)

sentiment_recommendation_agreement = df["recommendation_sentiment_agrees"].mean()

print(
    "Strict agreement rate "
    "(Recommended=positive, Not Recommended=negative): "
    f"{sentiment_recommendation_agreement:.1%}"
)

Strict agreement rate (Recommended=positive, Not Recommended=negative): 66.6%


The strict agreement rate is intentionally conservative because neutral text can still be paired with a valid recommendation. Short comments like `good`, `10/10`, jokes, sarcasm, or blank reviews may not carry enough text signal for VADER to classify strongly. For scoring, use sentiment as a supporting signal alongside recommendation rate, playtime, review volume, and review longevity.

In [21]:
sentiment_by_recommendation_summary = df.groupby("recommendation").agg(
    review_count=("review", "count"),
    average_sentiment=("sentiment_score", "mean"),
    median_sentiment=("sentiment_score", "median"),
    positive_share=("sentiment_label", lambda values: (values == "positive").mean()),
    neutral_share=("sentiment_label", lambda values: (values == "neutral").mean()),
    negative_share=("sentiment_label", lambda values: (values == "negative").mean()),
).round(3)

sentiment_by_recommendation_summary

,review_count,average_sentiment,median_sentiment,positive_share,neutral_share,negative_share
recommendation,,,,,,
Not Recommended,4223,-0.014,0.000,0.394,0.166,0.440
Recommended,10394,0.467,0.625,0.758,0.131,0.111


### Inspect VADER disagreement cases

The validation table shows that VADER is much better at identifying `Recommended` reviews than it is at identifying `Not Recommended` reviews. This is common in game reviews: negative recommendations often contain positive wording, mixed phrasing, or context such as `great game but crashes`, which lexicon models can score as positive.

In [22]:
positive_text_negative_recommendation = (
    df[
        df["recommendation"].eq("Not Recommended")
        & df["sentiment_label"].eq("positive")
    ]
    .sort_values("sentiment_score", ascending=False)
    [
        [
            "title",
            "recommendation",
            "sentiment_score",
            "sentiment_label",
            "hour_played",
            "helpful",
            "review",
        ]
    ]
)

positive_text_negative_recommendation.head(10)

,title,recommendation,sentiment_score,sentiment_label,hour_played,helpful,review
4357,Grand Theft Auto V,Not Recommended,0.9984,positive,134,0,So I'll start off with a simple opening. GTA h...
14240,The Elder Scrolls V: Skyrim Special Edition,Not Recommended,0.9978,positive,828,0,Product received for free. TL DR longer review...
10681,PLAYERUNKNOWN'S BATTLEGROUNDS,Not Recommended,0.9975,positive,39,0,I'm leaving a negative review until all the ba...
3965,The Elder Scrolls V: Skyrim Special Edition,Not Recommended,0.9972,positive,19,26,Product received for free. Big Skyrim fan sinc...
9033,The Elder Scrolls V: Skyrim Special Edition,Not Recommended,0.9972,positive,37,0,Product received for free. I first got Skyrim ...
5721,Rust,Not Recommended,0.9965,positive,1674,0,Used to be good. Really good but now its compl...
3290,The Elder Scrolls V: Skyrim Special Edition,Not Recommended,0.9960,positive,0,0,Product received for free. Got this game free ...
6325,ASTRONEER,Not Recommended,0.9960,positive,102,0,DO NOT BUY THIS GAME!!!Not until the LAZY deve...
4067,The Elder Scrolls V: Skyrim Special Edition,Not Recommended,0.9960,positive,0,0,Product received for free. Sure my playtime is...
9619,Rocket League®,Not Recommended,0.9955,positive,584,7,This game is a very popular one among its kind...


In [23]:
negative_recommendation_sentiment_counts = (
    df[df["recommendation"].eq("Not Recommended")]
    .groupby("sentiment_label")
    .agg(
        review_count=("review", "count"),
        median_sentiment=("sentiment_score", "median"),
        median_hours=("hour_played", "median"),
        median_helpful=("helpful", "median"),
    )
    .sort_index()
)

negative_recommendation_sentiment_counts

,review_count,median_sentiment,median_hours,median_helpful
sentiment_label,,,,
negative,1860,-0.5574,104.0,0.0
neutral,701,0.0000,107.0,0.0
positive,1662,0.5994,82.0,0.0


Conclusion: for scoring games, Steam `recommended_proportion` should remain the primary approval signal because it is the player's final judgement. VADER sentiment is still useful, but mainly as a supporting text-quality signal: it helps surface games where the written review tone is especially positive or negative, and it provides drill-down context for the dashboard.

## Where scaling is used

Scaling is not used inside VADER. VADER analyses the raw review text and returns sentiment scores directly. Scaling enters later in two places:

1. During game-level sentiment construction, `log1p(helpful)` is used as a soft review credibility weight. This gives more influence to reviews that other players marked helpful, while preventing one highly upvoted review from dominating the game score.
2. During game-level comparison and visualisation, selected skewed features are log-transformed before scaling so that very large games do not flatten the rest of the dataset.

The first of these happens before the game-level aggregation, so we check the review-level helpful vote distribution here.

In [24]:
helpful_distribution_summary = df["helpful"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)
helpful_distribution_summary["skew"] = df["helpful"].skew()
helpful_distribution_summary.round(3)

count    14617.000
mean         3.716
std         58.740
min          0.000
50%          0.000
75%          0.000
90%          0.000
95%          1.000
99%         15.000
max       2464.000
skew        25.849
Name: helpful, dtype: float64

In [25]:
import plotly.express as px

helpful_weight_check = df[["helpful"]].copy()
helpful_weight_check["log1p_helpful"] = np.log1p(helpful_weight_check["helpful"])

helpful_weight_long = helpful_weight_check.melt(
    value_vars=["helpful", "log1p_helpful"],
    var_name="feature",
    value_name="value",
)

fig = px.box(
    helpful_weight_long,
    x="feature",
    y="value",
    points="outliers",
    title="Helpful Votes Before and After log1p Transformation",
)
fig.show()

`helpful` is highly concentrated: most reviews have few or no helpful votes, while a small number receive many. The `log1p` transform keeps the ordering of reviews but compresses extreme values, making it suitable as a credibility weight for game-level sentiment aggregation.

## Aggregate from reviews to games

The columns below turn review behaviour into game-level signals: popularity/confidence, recommendation strength, engagement, longevity, helpful review engagement, and sentiment. Funny-derived aggregate features are excluded because the EDA above shows they duplicate the helpful vote flag in this dataset.

The game-level sentiment score is deliberately not a plain average. It combines helpful-weighted VADER sentiment, the balance of positive versus negative sentiment labels, and a confidence shrinkage adjustment so games with few reviews are pulled toward the dataset baseline.

In [26]:
# Helpful votes act as a soft credibility signal.
# log1p prevents one highly upvoted review from dominating the whole game score.
df["sentiment_review_weight"] = 1 + np.log1p(df["helpful"])


def weighted_average(values, weights):
    if weights.sum() == 0:
        return values.mean()
    return np.average(values, weights=weights)


global_weighted_sentiment = weighted_average(
    df["sentiment_score"],
    df["sentiment_review_weight"],
)
global_label_balance = (
    df["sentiment_label"].eq("positive").mean()
    - df["sentiment_label"].eq("negative").mean()
)
global_label_balance

np.float64(0.44626120270917424)

Clearly large imbalance between Negative and Positive labelling in favour of positives

global_sentiment_baseline is the “normal sentiment level” of the whole dataset.The global sentiment score shows how  pos/neg the average text is but label imbalance tells us the skewing.

If dset is biased toward positive reviews, baseline inherits additional positivity.

The idea is that if a game has many reviews, its own sentiment score should be given more confidence. If a game has few reviews, pull its sentiment score closer to the overall baseline.

Confidence score to scale sentiment based on number of reviews for game. It then also determines how much of the global fallback is used.

In [27]:

global_sentiment_baseline = (
    0.7 * global_weighted_sentiment
    + 0.3 * global_label_balance
)

sentiment_by_game = df.groupby("title").apply(
    lambda group: pd.Series(
        {
            "helpful_weighted_sentiment": weighted_average(
                group["sentiment_score"],
                group["sentiment_review_weight"],
            ),
            "sentiment_positive_share": group["sentiment_label"].eq("positive").mean(),
            "sentiment_negative_share": group["sentiment_label"].eq("negative").mean(),
            "sentiment_neutral_share": group["sentiment_label"].eq("neutral").mean(),
        }
    ),
    include_groups=False,
)

sentiment_by_game["sentiment_label_balance"] = (
    sentiment_by_game["sentiment_positive_share"]
    - sentiment_by_game["sentiment_negative_share"]
)
sentiment_by_game["unshrunk_sentiment_score"] = (
    0.7 * sentiment_by_game["helpful_weighted_sentiment"]
    + 0.3 * sentiment_by_game["sentiment_label_balance"]
)

game_level_features = df.groupby("title").agg(
    review_count=("review", "count"),
    review_share=("review", lambda values: len(values) / total_reviews),
    recommended_proportion=("is_recommended", "mean"),
    average_hours_played=("hour_played", "mean"),
    median_hours_played=("hour_played", "median"),
    helpful_votes=("helpful", "sum"),
    helpful_review_proportion=("helpful", lambda values: (values > 0).mean()),
    average_helpful_per_review=("helpful", "mean"),
    average_review_length=("review_length", "mean"),
    raw_average_sentiment=("sentiment_score", "mean"),
    first_review=("date_posted", "min"),
    last_review=("date_posted", "max"),
    early_access_review_share=("is_early_access_review", "mean"),
)

game_level_features = game_level_features.join(sentiment_by_game)

game_level_features["helpful_vote_share"] = np.where(
    total_helpful_votes > 0,
    game_level_features["helpful_votes"] / total_helpful_votes,
    0,
)

game_level_features["review_span_days"] = (
    game_level_features["last_review"] - game_level_features["first_review"]
).dt.days.fillna(0).astype(int)

# here i used a bayesian style prior review count as baseline for trusting game's sentiment
prior_review_strength = 25
game_level_features["sentiment_confidence"] = (
    game_level_features["review_count"]
    / (game_level_features["review_count"] + prior_review_strength)
)
# weighted average of game and global baseline
game_level_features["game_sentiment_score"] = (
    game_level_features["sentiment_confidence"]
    * game_level_features["unshrunk_sentiment_score"]
    + (1 - game_level_features["sentiment_confidence"])
    * global_sentiment_baseline
)

game_level_features = (
    game_level_features[
        [
            "review_count",
            "review_share",
            "recommended_proportion",
            "average_hours_played",
            "median_hours_played",
            "review_span_days",
            "helpful_vote_share",
            "helpful_review_proportion",
            "average_helpful_per_review",
            "raw_average_sentiment",
            "helpful_weighted_sentiment",
            "sentiment_label_balance",
            "unshrunk_sentiment_score",
            "sentiment_confidence",
            "game_sentiment_score",
            "sentiment_positive_share",
            "sentiment_negative_share",
            "sentiment_neutral_share",
            "average_review_length",
            "first_review",
            "last_review",
            "early_access_review_share",
        ]
    ]
    .sort_values(["recommended_proportion", "review_count"], ascending=False)
    .reset_index()
)

game_level_features

,title,review_count,review_share,recommended_proportion,average_hours_played,median_hours_played,review_span_days,helpful_vote_share,helpful_review_proportion,average_helpful_per_review,...,unshrunk_sentiment_score,sentiment_confidence,game_sentiment_score,sentiment_positive_share,sentiment_negative_share,sentiment_neutral_share,average_review_length,first_review,last_review,early_access_review_share
0,Euro Truck Simulator 2,331,0.022645,1.000000,206.749245,96.0,1621,0.039919,0.223565,6.549849,...,0.605427,0.929775,0.587959,0.842900,0.054381,0.102719,206.806647,2013-02-15,2017-07-25,0.000000
1,Terraria,239,0.016351,1.000000,430.765690,283.0,2764,0.051224,0.171548,11.640167,...,0.641006,0.905303,0.614082,0.836820,0.075314,0.087866,243.058577,2011-05-31,2018-12-24,0.000000
2,Subnautica,204,0.013956,1.000000,92.303922,60.0,1369,0.090352,0.132353,24.053922,...,0.572130,0.890830,0.548610,0.843137,0.107843,0.049020,375.970588,2015-04-16,2019-01-14,0.200980
3,RimWorld,184,0.012588,1.000000,248.711957,130.5,910,0.049623,0.032609,14.646739,...,0.299303,0.880383,0.306167,0.673913,0.173913,0.152174,237.157609,2016-07-15,2019-01-11,0.940217
4,Stardew Valley,183,0.012520,1.000000,124.644809,91.0,1054,0.073283,0.131148,21.748634,...,0.686297,0.879808,0.646680,0.841530,0.049180,0.109290,322.601093,2016-02-26,2019-01-15,0.000000
5,Factorio,158,0.010809,1.000000,389.468354,176.5,949,0.063543,0.373418,21.841772,...,0.510266,0.863388,0.489285,0.746835,0.113924,0.139241,301.721519,2016-02-25,2018-10-01,1.000000
6,Beat Saber,10,0.000684,1.000000,57.300000,26.5,234,0.042349,1.000000,230.000000,...,0.432832,0.285714,0.378442,0.700000,0.100000,0.200000,637.300000,2018-05-04,2018-12-24,1.000000
7,ACE COMBAT™ 7: SKIES UNKNOWN,8,0.000547,1.000000,27.375000,25.5,5,0.043491,0.875000,295.250000,...,0.508548,0.242424,0.393501,0.750000,0.250000,0.000000,944.750000,2019-01-31,2019-02-05,0.000000
8,"Warhammer 40,000: Mechanicus",7,0.000479,1.000000,22.714286,20.0,78,0.007899,0.714286,61.285714,...,0.869545,0.218750,0.468874,0.857143,0.142857,0.000000,2398.714286,2018-11-15,2019-02-01,0.000000
9,Left 4 Dead 2,202,0.013820,0.995050,310.628713,82.5,2950,0.090462,0.173267,24.321782,...,0.520842,0.889868,0.502763,0.767327,0.089109,0.143564,120.712871,2010-12-20,2019-01-17,0.000000


## Preserve review drill-down by game

Review-level sentiment table and game reviews table. Games review table is a dicitonary with game titles as the key, and value as list of its reviews. Going to use in dashboard for lookup.

In [28]:
review_level_sentiment = df[
    [
        "title",
        "date_posted",
        "recommendation",
        "hour_played",
        "helpful",
        "funny",
        "review",
        "sentiment_neg",
        "sentiment_neu",
        "sentiment_pos",
        "sentiment_score",
        "sentiment_label",
        "review_length",
    ]
].copy()

game_reviews_table = {
    title: group[
        [
            "date_posted",
            "recommendation",
            "hour_played",
            "helpful",
            "funny",
            "review",
            "sentiment_score",
            "sentiment_label",
        ]
    ].to_dict(orient="records")
    for title, group in review_level_sentiment.groupby("title")
}

print(f"Game-level rows: {len(game_level_features)}")
print(f"Review-level rows: {len(review_level_sentiment)}")
print(f"Games with review drill-down: {len(game_reviews_table)}")

Game-level rows: 23
Review-level rows: 14617
Games with review drill-down: 23


In [29]:
import plotly.express as px

Is dataset dominated by a few games?

In [30]:
fig = px.bar(
    game_level_features.sort_values("review_share", ascending=True),
    x="review_share",
    y="title",
    orientation="h",
    title="Total Review Proportion by Game",
    hover_data=["review_share", "recommended_proportion"]
)
fig.show()

Recommendation Rate vs Reciew Count

Top right candidate is both a confident aggreagation due to high review volume and also high recommendation rate

In [31]:
fig = px.scatter(
    game_level_features,
    x="review_count",
    y="recommended_proportion",
    size="median_hours_played",
    color="game_sentiment_score",
    hover_name="title",
    log_x=True,
    title="Recommendation Rate vs Review Volume"
)
fig.show()

Does Text sentiment agree with recommendations?

A game with high recommendation but weaker text sentiment may need more review

In [32]:
fig = px.scatter(
    game_level_features,
    x="recommended_proportion",
    y="game_sentiment_score",
    size="review_count",
    color="sentiment_confidence",
    hover_name="title",
    title="Recommendation Rate vs Constructed Sentiment Score"
)
fig.show()

Player engagement vs sentiment

In [33]:
fig = px.scatter(
    game_level_features,
    x="median_hours_played",
    y="game_sentiment_score",
    size="review_count",
    color="recommended_proportion",
    hover_name="title",
    log_x=True,
    title="Player Engagement vs Sentiment"
)
fig.show()

Review count vs Review span

Try to identify whether a game has sustained appeal or short bursts

Long review span and high review count suggests dense lasting interests

In [34]:
fig = px.scatter(
    game_level_features,
    x="review_span_days",
    y="review_count",
    color="recommended_proportion",
    size="game_sentiment_score",
    hover_name="title",
    title="Review Longevity vs Review Volume"
)
fig.show()

## Game-level feature distributions

Before applying log transforms, inspect the distribution of the main game-level numeric features. `average_helpful_per_review` means the average number of helpful votes received by a review for that game: `total helpful votes for the game / review count`.

Count and duration variables can have long right tails, meaning a few games dominate the scale. Proportion and score features are already bounded, so they usually do not need log scaling even when their shapes are uneven. Because there are only 23 games, smoothed density curves are used as exploratory shape checks rather than precise population estimates.

In [35]:
distribution_columns = [
    "review_count",
    "review_share",
    "recommended_proportion",
    "average_hours_played",
    "median_hours_played",
    "review_span_days",
    "helpful_vote_share",
    "helpful_review_proportion",
    "average_helpful_per_review",
    "game_sentiment_score",
]

distribution_summary = game_level_features[distribution_columns].describe().T
distribution_summary["skew"] = game_level_features[distribution_columns].skew()
distribution_summary.round(3)

,count,mean,std,min,25%,50%,75%,max,skew
review_count,23.0,635.522,795.318,7.000,170.500,231.000,875.500,2539.000,1.457
review_share,23.0,0.043,0.054,0.000,0.012,0.016,0.060,0.174,1.457
recommended_proportion,23.0,0.850,0.188,0.429,0.652,0.986,1.000,1.000,-0.810
average_hours_played,23.0,195.800,148.656,22.714,54.900,153.671,299.784,494.692,0.478
median_hours_played,23.0,98.935,73.797,20.000,29.000,91.000,147.750,283.000,0.904
review_span_days,23.0,899.783,835.364,5.000,166.000,828.000,1343.000,2950.000,1.056
helpful_vote_share,23.0,0.043,0.037,0.000,0.011,0.041,0.068,0.116,0.609
helpful_review_proportion,23.0,0.281,0.342,0.000,0.072,0.131,0.298,1.000,1.348
average_helpful_per_review,23.0,40.003,75.464,0.000,0.556,11.427,24.188,295.250,2.682
game_sentiment_score,23.0,0.445,0.140,0.226,0.310,0.469,0.568,0.647,-0.042


### Log-scaling decision

Use log scaling only where it improves comparison without damaging interpretability. The table below separates skewed unbounded features from bounded proportions and already-normalised scores.

In [36]:
log_transform_decisions = pd.DataFrame(
    [
        {
            "feature": "review_count",
            "skew": distribution_summary.loc["review_count", "skew"],
            "use_log_scale": True,
            "reason": "Unbounded count with a long right tail; large games dominate raw scale.",
        },
        {
            "feature": "review_share",
            "skew": distribution_summary.loc["review_share", "skew"],
            "use_log_scale": False,
            "reason": "Bounded proportion; keep raw for direct interpretation of dataset share.",
        },
        {
            "feature": "recommended_proportion",
            "skew": distribution_summary.loc["recommended_proportion", "skew"],
            "use_log_scale": False,
            "reason": "Bounded 0-1 rate; log transform would reduce interpretability.",
        },
        {
            "feature": "average_hours_played",
            "skew": distribution_summary.loc["average_hours_played", "skew"],
            "use_log_scale": False,
            "reason": "Only mildly skewed here; median playtime is preferred for robust engagement.",
        },
        {
            "feature": "median_hours_played",
            "skew": distribution_summary.loc["median_hours_played", "skew"],
            "use_log_scale": True,
            "reason": "Duration feature with diminishing returns; log scale improves comparability.",
        },
        {
            "feature": "review_span_days",
            "skew": distribution_summary.loc["review_span_days", "skew"],
            "use_log_scale": False,
            "reason": "Only moderately skewed and directly interpretable as days; use raw min-max scaling for scoring.",
        },
        {
            "feature": "helpful_vote_share",
            "skew": distribution_summary.loc["helpful_vote_share", "skew"],
            "use_log_scale": False,
            "reason": "Bounded share of total helpful votes; keep raw for interpretability.",
        },
        {
            "feature": "helpful_review_proportion",
            "skew": distribution_summary.loc["helpful_review_proportion", "skew"],
            "use_log_scale": False,
            "reason": "Bounded 0-1 proportion; uneven shape does not require log scaling.",
        },
        {
            "feature": "average_helpful_per_review",
            "skew": distribution_summary.loc["average_helpful_per_review", "skew"],
            "use_log_scale": True,
            "reason": "Unbounded intensity measure with strong right skew; log scale limits influence of highly upvoted reviews.",
        },
        {
            "feature": "game_sentiment_score",
            "skew": distribution_summary.loc["game_sentiment_score", "skew"],
            "use_log_scale": False,
            "reason": "Constructed bounded sentiment score; already comparable across games.",
        },
    ]
)

log_transform_decisions["skew"] = log_transform_decisions["skew"].round(3)
log_transform_decisions

,feature,skew,use_log_scale,reason
0,review_count,1.457,True,Unbounded count with a long right tail; large ...
1,review_share,1.457,False,Bounded proportion; keep raw for direct interp...
2,recommended_proportion,-0.810,False,Bounded 0-1 rate; log transform would reduce i...
3,average_hours_played,0.478,False,Only mildly skewed here; median playtime is pr...
4,median_hours_played,0.904,True,Duration feature with diminishing returns; log...
5,review_span_days,1.056,False,Only moderately skewed and directly interpreta...
6,helpful_vote_share,0.609,False,Bounded share of total helpful votes; keep raw...
7,helpful_review_proportion,1.348,False,Bounded 0-1 proportion; uneven shape does not ...
8,average_helpful_per_review,2.682,True,Unbounded intensity measure with strong right ...
9,game_sentiment_score,-0.042,False,Constructed bounded sentiment score; already c...


In [37]:
from scipy.stats import gaussian_kde


def build_kde_frame(data, columns, value_column="value"):
    kde_rows = []

    for column in columns:
        values = data[column].dropna().astype(float)
        if len(values) < 2:
            continue

        x_min = values.min()
        x_max = values.max()

        if x_min == x_max:
            x_grid = np.array([x_min])
            density = np.array([1.0])
        else:
            padding = (x_max - x_min) * 0.08
            x_grid = np.linspace(x_min - padding, x_max + padding, 200)
            density = gaussian_kde(values)(x_grid)

        kde_rows.append(
            pd.DataFrame(
                {
                    "feature": column,
                    value_column: x_grid,
                    "density": density,
                }
            )
        )

    return pd.concat(kde_rows, ignore_index=True)


raw_kde = build_kde_frame(game_level_features, distribution_columns)

fig = px.line(
    raw_kde,
    x="value",
    y="density",
    facet_col="feature",
    facet_col_wrap=3,
    title="Smoothed Game-Level Feature Distributions",
)
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None, showticklabels=False)
fig.update_traces(line_width=3)
fig.update_layout(height=900, showlegend=False)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig.show()

The log-scaled view below is an EDA check before scoring. It focuses on the unbounded features that will later be log-transformed for ranking: `review_count`, `median_hours_played`, and `average_helpful_per_review`. `review_span_days` is kept on its raw scale because its skew is moderate and the unit remains interpretable.

In [38]:
skewed_columns = [
    "review_count",
    "median_hours_played",
    "average_helpful_per_review",
]

log_distribution_features = game_level_features.copy()
for column in skewed_columns:
    log_distribution_features[f"log1p_{column}"] = np.log1p(
        log_distribution_features[column]
    )

log_kde_columns = [f"log1p_{column}" for column in skewed_columns]
log_kde = build_kde_frame(
    log_distribution_features,
    log_kde_columns,
    value_column="log1p_value",
)
log_kde["feature"] = log_kde["feature"].str.replace("log1p_", "", regex=False)

fig = px.line(
    log_kde,
    x="log1p_value",
    y="density",
    facet_col="feature",
    facet_col_wrap=3,
    title="Smoothed Log-Scaled Distributions for Selected Scoring Features",
)
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None, showticklabels=False)
fig.update_traces(line_width=3)
fig.update_layout(height=700, showlegend=False)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig.show()

Based on these distributions, the ranking step will log-transform only `review_count`, `median_hours_played`, and `average_helpful_per_review`. `review_span_days` is retained as raw days and then min-max scaled, because its skew is moderate and the original unit is useful for interpretation. Bounded proportions such as recommendation rate, review share, helpful review share, and constructed sentiment are not log-scaled.

### EDA-to-scoring scaling plan

The visual EDA above comes before the ranking model and determines which features are transformed. The scoring model below uses this plan: apply `log1p` to skewed unbounded features, leave bounded proportions raw, then min-max scale model inputs before combining them.

In [39]:
scaling_plan = pd.DataFrame(
    [
        {
            "scoring_input": "evidence_score",
            "source_feature": "review_count",
            "transformation": "log1p then rank scale",
            "eda_reason": "Review volume is a right-skewed count; ranking is robust to outliers while preserving relative evidence strength.",
        },
        {
            "scoring_input": "engagement_score",
            "source_feature": "median_hours_played",
            "transformation": "log1p then rank scale",
            "eda_reason": "Playtime is an unbounded duration; rank scaling avoids one very high-playtime game dominating.",
        },
        {
            "scoring_input": "longevity_score",
            "source_feature": "review_span_days",
            "transformation": "rank scale",
            "eda_reason": "Review span is a magnitude feature used relatively across games, while retaining raw days elsewhere for interpretation.",
        },
        {
            "scoring_input": "review_quality_score",
            "source_feature": "average_helpful_per_review",
            "transformation": "log1p then rank scale",
            "eda_reason": "Helpful intensity is highly skewed; rank scaling is robust to outlier reviews.",
        },
        {
            "scoring_input": "approval_score",
            "source_feature": "recommended_proportion",
            "transformation": "use directly",
            "eda_reason": "Already a meaningful 0-1 rate; 95% approval should remain 95% approval.",
        },
        {
            "scoring_input": "approval_score",
            "source_feature": "game_sentiment_score",
            "transformation": "use directly",
            "eda_reason": "Constructed bounded sentiment score; kept as supporting text evidence without rescaling its absolute meaning.",
        },
        {
            "scoring_input": "review_quality_score",
            "source_feature": "helpful_review_proportion",
            "transformation": "use directly",
            "eda_reason": "Already a meaningful 0-1 proportion.",
        },
    ]
)

scaling_plan

,scoring_input,source_feature,transformation,eda_reason
0,evidence_score,review_count,log1p then rank scale,Review volume is a right-skewed count; ranking...
1,engagement_score,median_hours_played,log1p then rank scale,Playtime is an unbounded duration; rank scalin...
2,longevity_score,review_span_days,rank scale,Review span is a magnitude feature used relati...
3,review_quality_score,average_helpful_per_review,log1p then rank scale,Helpful intensity is highly skewed; rank scali...
4,approval_score,recommended_proportion,use directly,Already a meaningful 0-1 rate; 95% approval sh...
5,approval_score,game_sentiment_score,use directly,Constructed bounded sentiment score; kept as s...
6,review_quality_score,helpful_review_proportion,use directly,Already a meaningful 0-1 proportion.


## Game acquisition ranking

The game-level feature table now needs to answer the client question: **which games should GameVault investigate first for licensing or publishing deals?**

There is no labelled outcome such as `successful acquisition`, `future revenue`, or `deal completed`, so this is not a supervised prediction model. Instead, it is a transparent prioritisation score built from the signals available in the dataset.

The score is designed around five business questions:

1. **Do players like the game?** Measured mainly by Steam recommendation rate, supported by review-text sentiment.
2. **Do players spend meaningful time in the game?** Measured by median hours played.
3. **Is there enough evidence to trust the signal?** Measured by review count.
4. **Has interest lasted over time?** Measured by the span between first and last review.
5. **Are the reviews useful to other players?** Measured by helpful review engagement.

Sentiment is included, but it is not allowed to dominate. The validation step showed VADER identifies positive recommended reviews well, but it is weaker on `Not Recommended` reviews because negative recommendations often contain mixed positive wording such as “great game but broken”. For that reason, Steam `recommended_proportion` remains the primary approval signal, while VADER sentiment acts as supporting text evidence.

### Scoring rationale

The final score is designed to answer: **which games should GameVault investigate first?**

`final_score = 35% approval + 25% engagement + 20% evidence + 15% longevity + 5% review quality`

- **35% approval_score**: This receives the highest weight because player approval is the core value signal. If players do not broadly recommend the game, it is unlikely to be a strong licensing or publishing opportunity. The score is mostly Steam recommendation rate, with a smaller VADER sentiment contribution because sentiment adds review-text context but is less reliable for negative recommendations.

- **25% engagement_score**: This receives the second-highest weight because GameVault wants games with sticky player behaviour. A game can be highly rated but shallow if players do not spend much time in it. Median playtime captures depth of play: whether players actually stay with the game.

- **20% evidence_score**: This measures confidence. A game with many reviews gives stronger evidence that its approval and engagement signals are reliable. This matters because GameVault is a small publisher and cannot afford to prioritise games based on very thin evidence.

- **15% longevity_score**: This measures sustained market attention. It differs from engagement: **engagement is how deeply players play**, while **longevity is how long the game continues attracting review activity over time**. Longevity helps separate lasting appeal from short-term buzz.

- **5% review_quality_score**: Helpful votes are useful, but indirect. They suggest whether reviews are useful to other players, so this is included as a small credibility signal rather than a main driver of acquisition value.

Rates and proportions keep their absolute meaning. Magnitude features such as review count, playtime, review span, and helpful intensity are rank-scaled because they are relative evidence signals and can be distorted by outliers.

### Risk flag descriptions

Risk flags are shown separately from the score. They do **not** reduce `final_score`; they tell GameVault what to investigate before making a licensing or publishing decision.

- **`risk_flag_low_review_count`**: `review_count < 25`. The game has limited review evidence, so its score may be less reliable.
- **`risk_flag_short_review_span`**: `review_span_days < 90`. Reviews are concentrated in a short time window, which may indicate short-term buzz rather than sustained demand.
- **`risk_flag_mixed_approval`**: `recommended_proportion < 0.70`. Player approval is mixed, so the game needs closer qualitative review before prioritisation.
- **`risk_flag_count`**: the number of active risk flags for the game. A higher count does not automatically disqualify a game, but it increases the amount of due diligence required.

In business terms: **the score creates the shortlist; the risk flags guide the follow-up questions.**

In [40]:
ranking_features = game_level_features.copy()


def rank_scale(series):
    if series.nunique(dropna=True) <= 1:
        return pd.Series(0.0, index=series.index)
    return (series.rank(method="average") - 1) / (len(series) - 1)

ranking_features["log_review_count"] = np.log1p(ranking_features["review_count"])
ranking_features["log_median_hours_played"] = np.log1p(
    ranking_features["median_hours_played"]
)
ranking_features["log_average_helpful_per_review"] = np.log1p(
    ranking_features["average_helpful_per_review"]
)

ranking_features["review_count_score"] = rank_scale(ranking_features["log_review_count"])
ranking_features["median_playtime_score"] = rank_scale(
    ranking_features["log_median_hours_played"]
)
ranking_features["review_span_score"] = rank_scale(ranking_features["review_span_days"])
ranking_features["helpful_intensity_score"] = rank_scale(
    ranking_features["log_average_helpful_per_review"]
)


ranking_features["approval_score"] = (
    0.85 * ranking_features["recommended_proportion"]
    + 0.15 * ranking_features["game_sentiment_score"]
)
ranking_features["evidence_score"] = ranking_features["review_count_score"]
ranking_features["engagement_score"] = ranking_features["median_playtime_score"]
ranking_features["longevity_score"] = ranking_features["review_span_score"]
ranking_features["review_quality_score"] = (
    0.60 * ranking_features["helpful_review_proportion"]
    + 0.40 * ranking_features["helpful_intensity_score"]
)

# Final opportunity score
ranking_features["final_score"] = (
    0.35 * ranking_features["approval_score"]
    + 0.25 * ranking_features["engagement_score"]
    + 0.20 * ranking_features["evidence_score"]
    + 0.15 * ranking_features["longevity_score"]
    + 0.05 * ranking_features["review_quality_score"]
)


ranking_features["risk_flag_low_review_count"] = ranking_features["review_count"] < 25
ranking_features["risk_flag_short_review_span"] = ranking_features["review_span_days"] < 90
ranking_features["risk_flag_mixed_approval"] = ranking_features["recommended_proportion"] < 0.70
ranking_features["risk_flag_count"] = ranking_features[
    [
        "risk_flag_low_review_count",
        "risk_flag_short_review_span",
        "risk_flag_mixed_approval",
    ]
].sum(axis=1)

ranking_features["opportunity_rank"] = (
    ranking_features["final_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

ranking_features["opportunity_tier"] = pd.qcut(
    ranking_features["final_score"],
    q=4,
    labels=["Low Priority", "Watchlist", "Strong Candidate", "Priority Target"],
)

ranking_features = ranking_features.sort_values("opportunity_rank")

ranking_columns = [
    "title",
    "opportunity_rank",
    "opportunity_tier",
    "final_score",
    "approval_score",
    "engagement_score",
    "evidence_score",
    "longevity_score",
    "review_quality_score",
    "review_count",
    "recommended_proportion",
    "game_sentiment_score",
    "median_hours_played",
    "review_span_days",
    "risk_flag_count",
]

ranking_features[ranking_columns].head(15).round(3)

,title,opportunity_rank,opportunity_tier,final_score,approval_score,engagement_score,evidence_score,longevity_score,review_quality_score,review_count,recommended_proportion,game_sentiment_score,median_hours_played,review_span_days,risk_flag_count
1,Terraria,1,Priority Target,0.848,0.942,1.000,0.545,0.955,0.321,239,1.000,0.614,283.0,2764,0
16,Rust,2,Priority Target,0.747,0.603,0.818,0.955,0.909,0.081,2291,0.664,0.256,171.0,1887,1
18,Grand Theft Auto V,3,Priority Target,0.742,0.579,0.909,0.909,0.818,0.147,2174,0.632,0.280,182.0,1405,1
0,Euro Truck Simulator 2,4,Priority Target,0.726,0.938,0.545,0.591,0.864,0.280,331,1.000,0.588,96.0,1621,0
15,PLAYERUNKNOWN'S BATTLEGROUNDS,5,Priority Target,0.722,0.621,0.955,1.000,0.409,0.105,2539,0.683,0.265,234.0,692,1
5,Factorio,6,Priority Target,0.698,0.923,0.864,0.227,0.591,0.497,158,1.000,0.489,176.5,949,0
9,Left 4 Dead 2,7,Strong Candidate,0.679,0.921,0.455,0.364,1.000,0.413,202,0.995,0.503,82.5,2950,0
20,Rocket League®,8,Strong Candidate,0.678,0.562,0.773,0.864,0.727,0.130,1773,0.606,0.314,165.0,1317,1
10,Slay the Spire,9,Strong Candidate,0.658,0.938,0.659,0.500,0.364,0.195,231,0.991,0.639,107.0,409,0
3,RimWorld,10,Strong Candidate,0.654,0.896,0.727,0.318,0.545,0.256,184,1.000,0.306,130.5,910,0


In [41]:
score_component_columns = [
    "approval_score",
    "engagement_score",
    "evidence_score",
    "longevity_score",
    "review_quality_score",
]

score_components_long = ranking_features.melt(
    id_vars=["title", "opportunity_rank", "opportunity_tier"],
    value_vars=score_component_columns,
    var_name="score_component",
    value_name="component_value",
)

top_ranked_titles = ranking_features.head(10)["title"].tolist()
score_components_top = score_components_long[
    score_components_long["title"].isin(top_ranked_titles)
]

fig = px.bar(
    score_components_top,
    x="component_value",
    y="title",
    color="score_component",
    orientation="h",
    title="Top Game Candidates: Score Component Breakdown",
    category_orders={"title": list(reversed(top_ranked_titles))},
)
fig.update_layout(height=650, xaxis_title="Scaled component score", yaxis_title="Game")
fig.show()

In [42]:
fig = px.scatter(
    ranking_features,
    x="engagement_score",
    y="approval_score",
    size="review_count",
    color="opportunity_tier",
    hover_name="title",
    hover_data=[
        "opportunity_rank",
        "final_score",
        "evidence_score",
        "longevity_score",
        "review_quality_score",
        "risk_flag_count",
    ],
    title="Candidate Map: Approval vs Engagement",
)
fig.update_layout(xaxis_title="Engagement score", yaxis_title="Approval score")
fig.show()

In [43]:
game_level_features = game_level_features.merge(
    ranking_features[
        [
            "title",
            "opportunity_rank",
            "opportunity_tier",
            "final_score",
            "approval_score",
            "engagement_score",
            "evidence_score",
            "longevity_score",
            "review_quality_score",
            "risk_flag_low_review_count",
            "risk_flag_short_review_span",
            "risk_flag_mixed_approval",
            "risk_flag_count",
        ]
    ],
    on="title",
    how="left",
)

game_level_features = game_level_features.sort_values("opportunity_rank")
game_level_features.head(10)[
    [
        "title",
        "opportunity_rank",
        "opportunity_tier",
        "final_score",
        "review_count",
        "recommended_proportion",
        "median_hours_played",
        "review_span_days",
        "risk_flag_count",
    ]
]

,title,opportunity_rank,opportunity_tier,final_score,review_count,recommended_proportion,median_hours_played,review_span_days,risk_flag_count
1,Terraria,1,Priority Target,0.848068,239,1.000000,283.0,2764,0
16,Rust,2,Priority Target,0.746981,2291,0.664339,171.0,1887,1
18,Grand Theft Auto V,3,Priority Target,0.741879,2174,0.632015,182.0,1405,1
0,Euro Truck Simulator 2,4,Priority Target,0.726438,331,1.000000,96.0,1621,0
15,PLAYERUNKNOWN'S BATTLEGROUNDS,5,Priority Target,0.722415,2539,0.683340,234.0,692,1
5,Factorio,6,Priority Target,0.698026,158,1.000000,176.5,949,0
9,Left 4 Dead 2,7,Strong Candidate,0.679439,202,0.995050,82.5,2950,0
20,Rocket League®,8,Strong Candidate,0.678153,1773,0.605753,165.0,1317,1
10,Slay the Spire,9,Strong Candidate,0.657528,231,0.991342,107.0,409,0
3,RimWorld,10,Strong Candidate,0.653643,184,1.000000,130.5,910,0


## Save processed outputs

These files are optional, but useful for a dashboard because it can load the processed tables directly instead of recomputing every time.

In [44]:
game_level_features.to_csv("game_level_features.csv", index=False)
review_level_sentiment.to_csv("review_level_sentiment.csv", index=False)

# JSON needs dates converted to strings.
import json

with open("game_reviews_table.json", "w") as f:
    json.dump(game_reviews_table, f, default=str, indent=2)

print("Saved game_level_features.csv")
print("Saved review_level_sentiment.csv")
print("Saved game_reviews_table.json")

Saved game_level_features.csv
Saved review_level_sentiment.csv
Saved game_reviews_table.json
